# Live Hand Pose Estimation with OpenVINO™

This notebook demonstrates live hand pose estimation with OpenVINO, using [MediaPipe](https://developers.google.com/mediapipe) hand landmark models for detecting fine hand movements. The pipeline uses two models:

1. **Palm Detection** — locates hands in the frame
2. **Hand Landmark** — detects 21 keypoints per hand (wrist, finger joints, and fingertips)

This enables tracking of fine hand movements including individual finger positions. Final part of this notebook shows live inference results from a webcam. Additionally, you can also upload a video file.

> **NOTE**: To use a webcam, you must run this Jupyter notebook on a computer with a webcam. If you run on a server, the webcam will not work. However, you can still do inference on a video in the final step.

#### Table of contents:

- [Imports](#Imports)
- [The model](#The-model)
    - [Download and convert the models](#Download-and-convert-the-models)
    - [Load the models](#Load-the-models)
- [Processing](#Processing)
    - [Palm Detection Decoder](#Palm-Detection-Decoder)
    - [Hand Landmark Processing](#Hand-Landmark-Processing)
    - [Process Results](#Process-Results)
    - [Draw Hand Overlays](#Draw-Hand-Overlays)
    - [Main Processing Function](#Main-Processing-Function)
- [Run](#Run)
    - [Run Live Hand Pose Estimation](#Run-Live-Hand-Pose-Estimation)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/pose-estimation-webcam/pose-estimation.ipynb" />

In [ ]:
%pip install -q "openvino>=2023.1.0" opencv-python tqdm

## Imports
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import collections
import time
from pathlib import Path

import cv2
import numpy as np
from IPython import display
import openvino as ov

# Fetch `notebook_utils` module
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )

    open("notebook_utils.py", "w").write(r.text)

import notebook_utils as utils

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("pose-estimation.ipynb")

## The model
[back to top ⬆️](#Table-of-contents:)

### Download and convert the models
[back to top ⬆️](#Table-of-contents:)

We use two [MediaPipe](https://developers.google.com/mediapipe) models for hand pose estimation:

- **Palm Detection Full** — an SSD-based model that detects palm bounding boxes and keypoints in the full frame
- **Hand Landmark Full** — detects 21 hand keypoints (wrist, each finger's MCP/PIP/DIP/TIP joints) from a cropped hand region

Both models are downloaded as TensorFlow Lite files and converted to OpenVINO IR format using `ov.convert_model`.

In [ ]:
# Directory for downloaded models.
base_model_dir = Path("model")
base_model_dir.mkdir(exist_ok=True)

# MediaPipe hand model URLs
palm_det_url = "https://storage.googleapis.com/mediapipe-assets/palm_detection_full.tflite"
hand_lm_url = "https://storage.googleapis.com/mediapipe-assets/hand_landmark_full.tflite"

palm_det_tflite = base_model_dir / "palm_detection_full.tflite"
hand_lm_tflite = base_model_dir / "hand_landmark_full.tflite"

# Download TFLite models
utils.download_file(palm_det_url, palm_det_tflite.name, palm_det_tflite.parent)
utils.download_file(hand_lm_url, hand_lm_tflite.name, hand_lm_tflite.parent)

# Convert to OpenVINO IR format
palm_det_ir = base_model_dir / "palm_detection_full.xml"
hand_lm_ir = base_model_dir / "hand_landmark_full.xml"

if not palm_det_ir.exists():
    palm_ov_model = ov.convert_model(palm_det_tflite)
    ov.save_model(palm_ov_model, palm_det_ir)

if not hand_lm_ir.exists():
    hand_ov_model = ov.convert_model(hand_lm_tflite)
    ov.save_model(hand_ov_model, hand_lm_ir)

print(f"Palm detection model: {palm_det_ir}")
print(f"Hand landmark model: {hand_lm_ir}")

### Load the models
[back to top ⬆️](#Table-of-contents:)

Load both converted models and compile them for the selected device. Select device from dropdown list for running inference using OpenVINO.

In [ ]:
device = utils.device_widget()

device

In [ ]:
import openvino.properties.hint as hints

core = ov.Core()

# Compile palm detection model
palm_model = core.read_model(palm_det_ir)
palm_compiled = core.compile_model(
    model=palm_model,
    device_name=device.value,
    config={hints.performance_mode(): hints.PerformanceMode.LATENCY},
)

# Compile hand landmark model
hand_model = core.read_model(hand_lm_ir)
hand_compiled = core.compile_model(
    model=hand_model,
    device_name=device.value,
    config={hints.performance_mode(): hints.PerformanceMode.LATENCY},
)

palm_input = palm_compiled.input(0)
hand_input = hand_compiled.input(0)

print(f"Palm detection input shape: {list(palm_input.shape)}")
print(f"Hand landmark input shape:  {list(hand_input.shape)}")
print(f"Palm detection outputs: {[(o.any_name, list(o.shape)) for o in palm_compiled.outputs]}")
print(f"Hand landmark outputs:  {[(o.any_name, list(o.shape)) for o in hand_compiled.outputs]}")

The palm detection model takes a 192×192 RGB image and produces two outputs: bounding box regressors (2016 anchor predictions × 18 values) and classification scores. The hand landmark model takes a 224×224 cropped hand image and produces 21 3D keypoints (63 values), a hand presence flag, and a handedness score.

In [ ]:
print("Palm detection model:")
print(f"  Input:  {palm_input.any_name} {list(palm_input.shape)}")
for o in palm_compiled.outputs:
    print(f"  Output: {o.any_name} {list(o.shape)}")

print("\nHand landmark model:")
print(f"  Input:  {hand_input.any_name} {list(hand_input.shape)}")
for o in hand_compiled.outputs:
    print(f"  Output: {o.any_name} {list(o.shape)}")

### Palm Detection Decoder
[back to top ⬆️](#Table-of-contents:)

The palm detection model uses an SSD-style architecture with 2016 anchors across 4 feature map layers (strides 8, 16, 16, 16). The decoder generates anchors, decodes bounding boxes and keypoints from the raw model output, and applies non-maximum suppression to produce final palm detections.

In [ ]:
PALM_INPUT_SIZE = 192
HAND_INPUT_SIZE = 224


def generate_anchors(input_size=192):
    """Generate SSD anchors for MediaPipe palm detection model.

    Layers with the same stride are grouped so that anchors at each grid cell
    are interleaved across sub-layers, matching the model's expected order.
    """
    strides = [8, 16, 16, 16]
    anchors = []
    layer_id = 0
    while layer_id < len(strides):
        stride = strides[layer_id]
        # Count how many consecutive layers share this stride
        same_stride_count = 0
        while (layer_id + same_stride_count < len(strides)
               and strides[layer_id + same_stride_count] == stride):
            same_stride_count += 1
        grid_size = input_size // stride
        for y in range(grid_size):
            for x in range(grid_size):
                for _ in range(same_stride_count):  # sub-layers at this stride
                    for _ in range(2):  # 2 anchors per sub-layer
                        anchors.append([(x + 0.5) / grid_size, (y + 0.5) / grid_size])
        layer_id += same_stride_count
    return np.array(anchors, dtype=np.float32)


def nms(boxes, scores, iou_threshold=0.3):
    """Non-maximum suppression."""
    if len(boxes) == 0:
        return []

    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []

    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        intersection = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        iou = intersection / (areas[i] + areas[order[1:]] - intersection + 1e-6)

        remaining = np.where(iou <= iou_threshold)[0]
        order = order[remaining + 1]

    return keep


def decode_palm_detections(raw_boxes, raw_scores, anchors, input_size=192,
                           score_threshold=0.5, iou_threshold=0.3):
    """Decode palm detection model outputs into detection results.

    Each detection contains a bounding box, confidence score, and 7 keypoints
    (wrist center, finger MCPs, thumb CMC) in normalized [0, 1] coordinates.
    """
    scores = 1.0 / (1.0 + np.exp(-raw_scores.reshape(-1)))

    mask = scores >= score_threshold
    if not np.any(mask):
        return []

    filtered_scores = scores[mask]
    filtered_boxes = raw_boxes.reshape(-1, 18)[mask]
    filtered_anchors = anchors[mask]

    # Decode box center and size (offsets are in input pixel coords)
    cx = filtered_boxes[:, 0] / input_size + filtered_anchors[:, 0]
    cy = filtered_boxes[:, 1] / input_size + filtered_anchors[:, 1]
    w = filtered_boxes[:, 2] / input_size
    h = filtered_boxes[:, 3] / input_size

    # Convert to corner format
    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=-1)

    # Decode 7 keypoints
    keypoints = np.zeros((len(filtered_boxes), 7, 2))
    for k in range(7):
        keypoints[:, k, 0] = filtered_boxes[:, 4 + 2 * k] / input_size + filtered_anchors[:, 0]
        keypoints[:, k, 1] = filtered_boxes[:, 4 + 2 * k + 1] / input_size + filtered_anchors[:, 1]

    indices = nms(boxes, filtered_scores, iou_threshold)

    detections = []
    for i in indices:
        detections.append({
            "box": boxes[i],
            "score": filtered_scores[i],
            "keypoints": keypoints[i],
        })
    return detections


# Pre-generate anchors
anchors = generate_anchors(PALM_INPUT_SIZE)
print(f"Generated {len(anchors)} anchors")

## Processing
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Hand skeleton connections (21 keypoints, MediaPipe format)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),           # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),           # Index finger
    (5, 9), (9, 10), (10, 11), (11, 12),      # Middle finger
    (9, 13), (13, 14), (14, 15), (15, 16),    # Ring finger
    (13, 17), (17, 18), (18, 19), (19, 20),   # Pinky
    (0, 17),                                    # Palm base
]


def get_hand_crop(img, detection, scale_factor=2.6, target_size=224):
    """Extract a rotation-aware hand crop using palm detection keypoints.

    Uses the wrist (keypoint 0) and middle finger MCP (keypoint 2) to determine
    hand orientation, then creates a rotated crop for the landmark model.

    Returns the cropped image and the affine transform matrix used.
    """
    img_h, img_w = img.shape[:2]

    # Get wrist and middle finger MCP keypoints for orientation
    kp_wrist = detection["keypoints"][0] * np.array([img_w, img_h])
    kp_middle = detection["keypoints"][2] * np.array([img_w, img_h])

    # Compute rotation angle to orient hand upward
    dx = kp_middle[0] - kp_wrist[0]
    dy = kp_middle[1] - kp_wrist[1]
    rotation = np.degrees(np.arctan2(dx, -dy))

    # Compute crop center and size from bounding box
    box = detection["box"] * np.array([img_w, img_h, img_w, img_h])
    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2
    box_size = max(box[2] - box[0], box[3] - box[1]) * scale_factor

    # Shift center slightly toward fingers
    shift = box_size * 0.05
    cx += shift * np.sin(np.radians(rotation))
    cy -= shift * np.cos(np.radians(rotation))

    # Build affine transform: rotate around crop center then scale to target size
    cos_r = np.cos(np.radians(-rotation))
    sin_r = np.sin(np.radians(-rotation))
    scale = target_size / box_size

    M = np.array([
        [cos_r * scale, -sin_r * scale, target_size / 2 - (cx * cos_r - cy * sin_r) * scale],
        [sin_r * scale,  cos_r * scale, target_size / 2 - (cx * sin_r + cy * cos_r) * scale],
    ], dtype=np.float32)

    cropped = cv2.warpAffine(img, M, (target_size, target_size))
    return cropped, M


def transform_landmarks_to_image(landmarks, M):
    """Transform 21 hand landmarks from crop coordinates back to original image coordinates."""
    M_full = np.vstack([M, [0, 0, 1]])
    M_inv = np.linalg.inv(M_full)[:2]

    ones = np.ones((landmarks.shape[0], 1))
    pts = np.hstack([landmarks[:, :2], ones])
    original_pts = (M_inv @ pts.T).T

    result = np.copy(landmarks)
    result[:, 0] = original_pts[:, 0]
    result[:, 1] = original_pts[:, 1]
    return result

### Process Results
[back to top ⬆️](#Table-of-contents:)

The processing pipeline runs palm detection on the full frame, then for each detected palm, extracts a rotated hand crop and runs the hand landmark model to get 21 keypoints. Landmarks are transformed back to original image coordinates.

In [ ]:
def detect_palms(frame):
    """Run palm detection on a frame. Returns list of detections with boxes and keypoints."""
    # Preprocess: resize, BGR→RGB, normalize to [0, 1]
    input_img = cv2.resize(frame, (PALM_INPUT_SIZE, PALM_INPUT_SIZE))
    input_img = cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB)
    input_img = input_img.astype(np.float32) / 255.0

    # Handle NHWC vs NCHW input layout
    input_shape = list(palm_compiled.input(0).shape)
    if input_shape[-1] == 3:
        input_tensor = np.expand_dims(input_img, 0)
    else:
        input_tensor = np.expand_dims(input_img.transpose(2, 0, 1), 0)

    results = palm_compiled([input_tensor])

    # Identify outputs by shape: regressors have last dim 18, scores have last dim 1
    out0 = results[palm_compiled.output(0)]
    out1 = results[palm_compiled.output(1)]
    if out0.shape[-1] == 18:
        raw_boxes, raw_scores = out0, out1
    else:
        raw_boxes, raw_scores = out1, out0

    return decode_palm_detections(raw_boxes, raw_scores, anchors, PALM_INPUT_SIZE)


def detect_hand_landmarks(frame, detection):
    """Run hand landmark detection on a palm region. Returns 21 keypoints and confidence."""
    cropped, M = get_hand_crop(frame, detection, target_size=HAND_INPUT_SIZE)

    # Preprocess: BGR→RGB, normalize to [0, 1]
    input_img = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    input_img = input_img.astype(np.float32) / 255.0

    input_shape = list(hand_compiled.input(0).shape)
    if input_shape[-1] == 3:
        input_tensor = np.expand_dims(input_img, 0)
    else:
        input_tensor = np.expand_dims(input_img.transpose(2, 0, 1), 0)

    results = hand_compiled([input_tensor])

    # Model has 4 outputs: screen landmarks (63), hand flag (1), handedness (1), world landmarks (63).
    # Distinguish screen vs world landmarks by value range (screen is in pixels [0, 224]).
    landmarks = None
    hand_flag = None

    landmark_candidates = []
    for output in hand_compiled.outputs:
        data = results[output].squeeze()
        if data.size == 63:
            landmark_candidates.append(data.reshape(21, 3))
        elif data.size == 1 and hand_flag is None:
            hand_flag = 1.0 / (1.0 + np.exp(-float(data)))

    if not landmark_candidates or hand_flag is None:
        return None, 0.0

    # Pick screen landmarks (pixel-scale values) over world landmarks (meter-scale)
    if len(landmark_candidates) == 1:
        landmarks = landmark_candidates[0]
    else:
        landmarks = max(landmark_candidates, key=lambda lm: np.abs(lm[:, :2]).max())

    # Landmarks are in crop pixel coordinates — transform to original image
    landmarks = transform_landmarks_to_image(landmarks, M)
    return landmarks, float(hand_flag)


def process_frame(frame, palm_score_threshold=0.5, hand_score_threshold=0.5):
    """Full pipeline: detect palms, then detect hand landmarks for each palm."""
    detections = detect_palms(frame)

    all_landmarks = []
    for det in detections:
        if det["score"] < palm_score_threshold:
            continue
        landmarks, confidence = detect_hand_landmarks(frame, det)
        if landmarks is not None and confidence > hand_score_threshold:
            all_landmarks.append(landmarks)

    return all_landmarks

### Draw Hand Overlays
[back to top ⬆️](#Table-of-contents:)

Draw hand pose overlays on the image to visualize estimated hand landmarks. Each finger is drawn in a distinct color, with keypoints as circles and connections as lines.

In [ ]:
# Colors per finger group (BGR format)
THUMB_COLOR = (0, 0, 255)
INDEX_COLOR = (0, 128, 255)
MIDDLE_COLOR = (0, 255, 0)
RING_COLOR = (255, 128, 0)
PINKY_COLOR = (255, 0, 128)
PALM_COLOR = (180, 180, 180)

# One color per connection, matching HAND_CONNECTIONS order
CONNECTION_COLORS = (
    [THUMB_COLOR] * 4
    + [INDEX_COLOR] * 4
    + [MIDDLE_COLOR] * 4
    + [RING_COLOR] * 4
    + [PINKY_COLOR] * 4
    + [PALM_COLOR]
)

# One color per keypoint (0=wrist, 1-4=thumb, 5-8=index, etc.)
KEYPOINT_COLORS = (
    [PALM_COLOR]
    + [THUMB_COLOR] * 4
    + [INDEX_COLOR] * 4
    + [MIDDLE_COLOR] * 4
    + [RING_COLOR] * 4
    + [PINKY_COLOR] * 4
)


def draw_hand_landmarks(img, all_landmarks, connections=HAND_CONNECTIONS):
    """Draw hand landmarks and skeleton connections on the image."""
    if not all_landmarks:
        return img

    img_limbs = np.copy(img)

    for landmarks in all_landmarks:
        points = landmarks[:, :2].astype(np.int32)

        # Draw connections
        for idx, (i, j) in enumerate(connections):
            cv2.line(img_limbs, tuple(points[i]), tuple(points[j]),
                     CONNECTION_COLORS[idx], 2, cv2.LINE_AA)

        # Draw keypoints
        for idx, pt in enumerate(points):
            cv2.circle(img, tuple(pt), 4, KEYPOINT_COLORS[idx], -1, cv2.LINE_AA)
            cv2.circle(img, tuple(pt), 4, (255, 255, 255), 1, cv2.LINE_AA)

    cv2.addWeighted(img, 0.4, img_limbs, 0.6, 0, dst=img)
    return img

### Main Processing Function
[back to top ⬆️](#Table-of-contents:)

Run hand pose estimation on the specified source. Either a webcam or a video file.

In [ ]:
def run_hand_estimation(source=0, flip=False, use_popup=False, skip_first_frames=0):
    """Main processing function to run hand pose estimation."""
    player = None
    try:
        player = utils.VideoPlayer(source, flip=flip, fps=30, skip_first_frames=skip_first_frames)
        player.start()
        if use_popup:
            title = "Press ESC to Exit"
            cv2.namedWindow(title, cv2.WINDOW_GUI_NORMAL | cv2.WINDOW_AUTOSIZE)

        processing_times = collections.deque()

        while True:
            frame = player.next()
            if frame is None:
                print("Source ended")
                break

            # Limit frame size for performance.
            scale = 1280 / max(frame.shape)
            if scale < 1:
                frame = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

            start_time = time.time()
            all_landmarks = process_frame(frame)
            stop_time = time.time()

            frame = draw_hand_landmarks(frame, all_landmarks)

            processing_times.append(stop_time - start_time)
            if len(processing_times) > 200:
                processing_times.popleft()

            _, f_width = frame.shape[:2]
            processing_time = np.mean(processing_times) * 1000
            fps = 1000 / processing_time
            cv2.putText(
                frame,
                f"Inference time: {processing_time:.1f}ms ({fps:.1f} FPS)",
                (20, 40),
                cv2.FONT_HERSHEY_COMPLEX,
                f_width / 1000,
                (0, 0, 255),
                1,
                cv2.LINE_AA,
            )

            if use_popup:
                cv2.imshow(title, frame)
                key = cv2.waitKey(1)
                if key == 27:
                    break
            else:
                _, encoded_img = cv2.imencode(".jpg", frame, params=[cv2.IMWRITE_JPEG_QUALITY, 90])
                i = display.Image(data=encoded_img)
                display.clear_output(wait=True)
                display.display(i)
    except KeyboardInterrupt:
        print("Interrupted")
    except RuntimeError as e:
        print(e)
    finally:
        if player is not None:
            player.stop()
        if use_popup:
            cv2.destroyAllWindows()

## Run
[back to top ⬆️](#Table-of-contents:)

### Run Live Hand Pose Estimation
[back to top ⬆️](#Table-of-contents:)

Use a webcam as the video input. By default, the primary webcam is set with `source=0`. If you have multiple webcams, each one will be assigned a consecutive number starting at 0. Set `flip=True` when using a front-facing camera. Some web browsers, especially Mozilla Firefox, may cause flickering. If you experience flickering, set `use_popup=True`.

> **NOTE**: To use this notebook with a webcam, you need to run the notebook on a computer with a webcam. If you run the notebook on a server (for example, Binder), the webcam will not work. Popup mode may not work if you run this notebook on a remote computer (for example, Binder).

If you do not have a webcam, you can still run this demo with a video file. Any [format supported by OpenCV](https://docs.opencv.org/4.5.1/dd/d43/tutorial_py_video_display.html) will work. You can skip first `N` frames to fast forward video.

Run the hand pose estimation:

In [ ]:
USE_WEBCAM = True
cam_id = 0
video_file = Path("store-aisle-detection.mp4")
video_url = "https://storage.openvinotoolkit.org/data/test_data/videos/store-aisle-detection.mp4"
source = cam_id if USE_WEBCAM else video_file

if not USE_WEBCAM and not Path(video_file).exists():
    utils.download_file(video_url)

additional_options = {"skip_first_frames": 500} if not USE_WEBCAM else {}
run_hand_estimation(source=source, flip=isinstance(source, int), use_popup=False, **additional_options)